# Hyperparameter Tuning & Imbalance Handling

Tune 3 model có F1-macro cao nhất từ baseline: **Random Forest**, **XGBoost**, **LightGBM**.

**Phương pháp:**
- `RandomizedSearchCV` trên 20% subsample (tiết kiệm thời gian)
- 3-fold Stratified CV, optimize theo `f1_macro`
- Imbalance handling tích hợp qua `class_weight` / `sample_weight`

In [1]:
import os
import sys
import time
import json
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import lightgbm as lgb

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))
sys.path.append(os.path.abspath(".."))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [2]:
# --- STEP 1: Load data + tạo subsample ---
print("=" * 70)
print("STEP 1: Loading data & creating subsample for tuning")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Full training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

# Lấy 20% training data để tune (giữ tỉ lệ class)
from sklearn.model_selection import train_test_split

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=0.2,
    random_state=42,
    stratify=y_train,
)

print(f"Subsample for tuning: {len(X_tune):,} samples ({len(X_tune)/len(X_train)*100:.0f}%)")
print(f"\nClass distribution in subsample:")
print(y_tune.value_counts().to_string())

STEP 1: Loading data & creating subsample for tuning
Loaded splits from disk.
Full training set: 2,016,609 samples
Test set: 504,153 samples
Subsample for tuning: 403,321 samples (20%)

Class distribution in subsample:
Attack Type
BENIGN                        335208
DoS                            30999
DDoS                           20482
PortScan                       14511
Brute Force                     1464
Bot                              312
Web Attack � Brute Force         235
Web Attack � XSS                 104
Web Attack � Sql Injection         4
Heartbleed                         2


In [3]:
# --- STEP 2: Setup chung ---
print("=" * 70)
print("STEP 2: CV & scoring setup")
print("=" * 70)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
N_ITER = 30        # số lần random search
RANDOM_STATE = 42

# Lưu kết quả tuning
tuning_results = {}

print(f"CV folds: 3")
print(f"Random search iterations: {N_ITER}")
print(f"Scoring: f1_macro")

STEP 2: CV & scoring setup
CV folds: 3
Random search iterations: 30
Scoring: f1_macro


In [4]:
# --- STEP 3: Tune Random Forest ---
print("\n" + "=" * 70)
print("STEP 3: Tuning Random Forest")
print("=" * 70)

rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced", "balanced_subsample", None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=N_ITER,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1,
)

t0 = time.time()
rf_search.fit(X_tune, y_tune)
rf_tune_time = time.time() - t0

print(f"\nTuning completed in {rf_tune_time:.1f}s")
print(f"Best CV F1-macro: {rf_search.best_score_:.4f}")
print(f"Best params: {rf_search.best_params_}")

tuning_results["Random Forest"] = {
    "best_params": rf_search.best_params_,
    "best_cv_score": rf_search.best_score_,
    "tune_time": rf_tune_time,
}


STEP 3: Tuning Random Forest
Fitting 3 folds for each of 30 candidates, totalling 90 fits


/Users/anhoang/Documents/Projects/DAP391m/Project/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# --- STEP 4: Tune XGBoost ---
print("\n" + "=" * 70)
print("STEP 4: Tuning XGBoost")
print("=" * 70)

# Label encoding cho XGBoost
le = LabelEncoder()
y_tune_encoded = le.fit_transform(y_tune)

xgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

# Tính sample weights cho class imbalance
weights_tune = compute_sample_weight(class_weight="balanced", y=y_tune_encoded)

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    ),
    param_distributions=xgb_param_dist,
    n_iter=N_ITER,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=1,  # XGBoost đã dùng n_jobs=-1 bên trong
)

t0 = time.time()
xgb_search.fit(X_tune, y_tune_encoded, sample_weight=weights_tune)
xgb_tune_time = time.time() - t0

print(f"\nTuning completed in {xgb_tune_time:.1f}s")
print(f"Best CV F1-macro: {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")

tuning_results["XGBoost"] = {
    "best_params": xgb_search.best_params_,
    "best_cv_score": xgb_search.best_score_,
    "tune_time": xgb_tune_time,
}

In [ ]:
# --- STEP 5: Tune LightGBM ---
print("\n" + "=" * 70)
print("STEP 5: Tuning LightGBM")
print("=" * 70)

lgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "min_child_samples": [5, 10, 20, 50],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "class_weight": ["balanced", None],
}

lgb_search = RandomizedSearchCV(
    lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    param_distributions=lgb_param_dist,
    n_iter=N_ITER,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=1,  # LightGBM đã dùng n_jobs=-1 bên trong
)

t0 = time.time()
lgb_search.fit(X_tune, y_tune)
lgb_tune_time = time.time() - t0

print(f"\nTuning completed in {lgb_tune_time:.1f}s")
print(f"Best CV F1-macro: {lgb_search.best_score_:.4f}")
print(f"Best params: {lgb_search.best_params_}")

tuning_results["LightGBM"] = {
    "best_params": lgb_search.best_params_,
    "best_cv_score": lgb_search.best_score_,
    "tune_time": lgb_tune_time,
}

In [ ]:
# --- STEP 6: Retrain best models trên full training set ---
print("\n" + "=" * 70)
print("STEP 6: Retraining best models on full training data")
print("=" * 70)

# ── Random Forest (tuned) ──
print("\n── Random Forest (tuned) ──")
t0 = time.time()
rf_tuned = RandomForestClassifier(
    **rf_search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_tuned.fit(X_train, y_train)
rf_train_time = time.time() - t0
print(f"Training completed in {rf_train_time:.1f}s")

# ── XGBoost (tuned) ──
print("\n── XGBoost (tuned) ──")
le_full = LabelEncoder()
y_train_encoded = le_full.fit_transform(y_train)
weights_full = compute_sample_weight(class_weight="balanced", y=y_train_encoded)

t0 = time.time()
xgb_tuned = XGBClassifier(
    **xgb_search.best_params_,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=1,
)
xgb_tuned.fit(X_train, y_train_encoded, sample_weight=weights_full)
xgb_train_time = time.time() - t0
print(f"Training completed in {xgb_train_time:.1f}s")

# ── LightGBM (tuned) ──
print("\n── LightGBM (tuned) ──")
t0 = time.time()
lgb_tuned = lgb.LGBMClassifier(
    **lgb_search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgb_tuned.fit(X_train, y_train)
lgb_train_time = time.time() - t0
print(f"Training completed in {lgb_train_time:.1f}s")

In [ ]:
# --- STEP 7: Evaluate tuned models trên test set ---
print("\n" + "=" * 70)
print("STEP 7: Evaluating tuned models on test set")
print("=" * 70)

all_results = []

# ── Random Forest (tuned) ──
y_pred_rf = rf_tuned.predict(X_test)
y_proba_rf = rf_tuned.predict_proba(X_test)
rf_results = evaluate_model(
    y_test, y_pred_rf,
    model_name="Random Forest (tuned)",
    y_pred_proba=y_proba_rf,
    labels=rf_tuned.classes_.tolist(),
)
all_results.append(rf_results)

# ── XGBoost (tuned) ──
y_pred_xgb_enc = xgb_tuned.predict(X_test)
y_proba_xgb = xgb_tuned.predict_proba(X_test)
y_pred_xgb = le_full.inverse_transform(y_pred_xgb_enc)
xgb_results = evaluate_model(
    y_test, y_pred_xgb,
    model_name="XGBoost (tuned)",
    y_pred_proba=y_proba_xgb,
    labels=le_full.classes_.tolist(),
)
all_results.append(xgb_results)

# ── LightGBM (tuned) ──
y_pred_lgb = lgb_tuned.predict(X_test)
y_proba_lgb = lgb_tuned.predict_proba(X_test)
lgb_results = evaluate_model(
    y_test, y_pred_lgb,
    model_name="LightGBM (tuned)",
    y_pred_proba=y_proba_lgb,
    labels=lgb_tuned.classes_.tolist(),
)
all_results.append(lgb_results)

In [ ]:
# --- STEP 8: Confusion Matrices ---
print("\n" + "=" * 70)
print("STEP 8: Confusion Matrices (tuned models)")
print("=" * 70)

models_preds = [
    ("Random Forest (tuned)", y_pred_rf, rf_tuned.classes_.tolist()),
    ("XGBoost (tuned)", y_pred_xgb, le_full.classes_.tolist()),
    ("LightGBM (tuned)", y_pred_lgb, lgb_tuned.classes_.tolist()),
]

for name, y_pred, labels in models_preds:
    plot_confusion_matrix(
        y_true=y_test,
        y_pred=y_pred,
        labels=labels,
        model_name=name,
        normalize=True,
        figsize=(12, 10),
    )

In [ ]:
# --- STEP 9: Tuning Summary ---
print("\n" + "=" * 70)
print("STEP 9: Tuning Summary")
print("=" * 70)

# So sánh kết quả tuned
comparison = compare_models(all_results)
print("\nTuned Model Comparison:")
print(comparison.to_string(index=False))

# In best params
print("\n" + "=" * 70)
print("Best Hyperparameters")
print("=" * 70)
for model_name, info in tuning_results.items():
    print(f"\n{model_name}:")
    print(f"  CV F1-macro: {info['best_cv_score']:.4f}")
    print(f"  Tuning time: {info['tune_time']:.1f}s")
    for k, v in info["best_params"].items():
        print(f"  {k}: {v}")

# Lưu results ra file để notebook 09 dùng
import os
results_dir = os.path.join(str(Path.cwd()), "results")
os.makedirs(results_dir, exist_ok=True)

# Lưu tuned results
with open(os.path.join(results_dir, "tuned_results.pkl"), "wb") as f:
    pickle.dump({
        "all_results": all_results,
        "tuning_results": tuning_results,
        "best_params": {
            "Random Forest": rf_search.best_params_,
            "XGBoost": xgb_search.best_params_,
            "LightGBM": lgb_search.best_params_,
        },
    }, f)

print(f"\nResults saved to {results_dir}/tuned_results.pkl")